# Prompt Eval Workflow

- **Why?** - Running a prompt through an _evaluation pipeline_ and scoring it helps in tweaking the prompt against edge-cases.
- **Steps** - Draft a prompt -> Create an evaluation dataset (actual/generated inputs) -> Feed to LLM -> Grader (Score based on relevance/accuracy) --> Average of this is the score of the prompt/model/both

In [1]:
# Install dependencies
%pip install anthropic python-dotenv

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

# Converation History Helper Functions
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=0.5, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Dataset Generation

- Can use cheaper & faster models like `Haiku` to save cost and imitate user inputs

In [2]:
import json

def generate_dataset():
    messages = []
    prompt = '''
        Generate a evaluation dataset for a prompt evaluation workflow.
        The dataset will be used to evaluate the prompt that generates python, JSON, or regex specifically for AWS-related tasks.
        Generate an array of JSON for each representation of the dataset that required Python, JSON or regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "python/json/regex",
                "solution_criteria": "Key criteria for evaluating the solution"
            },
            ... additional tasks
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
        '''
    
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json") # Prefilled assistant message to guide the model to output JSON
    response = chat(messages, stop_sequences=["```"]) # Stop generation after the closing JSON code block
    return json.loads(response)

> Note: 
> - To have better results on grading, provide more context to the grader on how a good solution looks like.
> - While `task` in the generated dataset is enough for understanding the responses, we've additionally added `format` to assist code-based grader, and `solution_criteria` to assist model-based grader.

In [3]:
# Check the generated dataset

dataset = generate_dataset()
dataset

with open("evaluation_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## Running the evaluation

- `run_prompt`: Pass in a prompt to generate results
- `run_test_case`: Score the results from previous step
- `run_eval`: Get the results & evaluate from previous step
- **For Grading**:
    - **Model-based** - For quality, completeness, and safety checks
    - **Code-based** - Format, syntax, and validity checks

In [4]:
def run_prompt(test_case):
    '''
    Merges the prompt & test case input, and returns the output.
    '''
    test_prompt = f'''
    Please solve the following task:
    {test_case["task"]}
    ''' # This is the PROMPT WE ARE EVALUATING, which will be used in the evaluation workflow to generate the output for each test case.

    messages = []
    add_user_message(messages, test_prompt)
    output = chat(messages)
    return output

In [5]:
def run_test_case(test_case):
    '''
    Runs the test case by calling the run_prompt function and comparing the output to the expected output.
    '''
    output = run_prompt(test_case)

    # TODO: Grading
    # Checkout 2_Prompt_Evaluation/1_2_Grading_Strategies.ipynb for different grading strategies you can implement to compare the output and expected output and assign a score.
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }



In [6]:
def run_eval(dataset):
    '''
    Runs the evaluation by iterating through the dataset and running each test case by calling the run_test_case function.
    '''
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [7]:
# Read the dataset from the JSON file

with open("evaluation_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [8]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a solution to parse an AWS S3 bucket name and region from an S3 URI:\n\n```python\nimport re\nfrom typing import Tuple, Optional\nimport boto3\n\ndef parse_s3_uri(s3_uri: str) -> Tuple[str, str]:\n    \"\"\"\n    Parse an S3 URI and return the bucket name and region.\n    \n    Args:\n        s3_uri: Full S3 URI (e.g., 's3://my-bucket-name/path/to/object')\n    \n    Returns:\n        Tuple of (bucket_name, region)\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    # Validate and parse the S3 URI\n    match = re.match(r's3://([a-z0-9.-]+)(?:/.*)?$', s3_uri)\n    \n    if not match:\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}\")\n    \n    bucket_name = match.group(1)\n    \n    # Get the region using boto3\n    s3_client = boto3.client('s3')\n    try:\n        response = s3_client.get_bucket_location(Bucket=bucket_name)\n        region = response['LocationConstraint'] or 'us-east-1'\n